In [1]:
!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="nVAF7gO0EsO2lvrQaN50")
project = rf.workspace("ai-computer-vision-jhyu1").project("satellite-a1j5s-oysvh")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to satellite-1 in yolov8:: 100%|██████████| 622/622 [00:00<00:00, 7526.19it/s]


In [2]:
!pip install ultralytics

In [3]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 model (choose n/s/m/l/x depending on your GPU)
model = YOLO('yolov8s.pt')

# Train
model.train(data='/content/satellite-1/data.yaml', epochs=20, imgsz=640)

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/satellite-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7976c78ddd00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

# Gradio-Image


In [4]:
#Install dependencies (run this once)
!pip install ultralytics gradio -q

#Import libraries
from ultralytics import YOLO
import gradio as gr
from PIL import Image

#Load your trained model
model = YOLO('/content/runs/detect/train/weights/best.pt')  # adjust path if needed

#Define a prediction function for Gradio
def detect_objects(image):
    # Run YOLO prediction
    results = model.predict(image)
    # Extract the first result image with boxes drawn
    result_image = results[0].plot()  # numpy array with boxes
    # Convert to PIL Image so Gradio can display it
    return Image.fromarray(result_image)

#Build the Gradio interface
demo = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(type="filepath", label="Upload an image"),
    outputs=gr.Image(label="Detection result"),
    title="YOLOv8 Object Detection",
    description="Upload an image and see what YOLOv8 detects!",
)

#Launch the app
demo.launch(share=True)  # 'share=True' gives you a public link


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec25aacb32cdca32a3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Gradio-Video

In [5]:
!pip install ultralytics gradio ffmpeg-python -q
import ffmpeg
import os, tempfile, shutil
from ultralytics import YOLO
import gradio as gr

# Load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

def detect_video(video_path):
    #Buat nama file aman
    safe_name = os.path.basename(video_path).replace(" ", "_").replace("(", "").replace(")", "")
    safe_path = os.path.join("/content", safe_name)
    shutil.copy(video_path, safe_path)

    #Jalankan YOLO
    model.predict(
        source=safe_path,
        save=True,
        project='runs/detect',
        name='gradio_output',
        exist_ok=True
    )

    #Cari hasil .avi atau .mp4
    folder = 'runs/detect/gradio_output'
    output_video = None
    for f in os.listdir(folder):
        if f.endswith('.avi') or f.endswith('.mp4'):
            output_video = os.path.join(folder, f)
            break

    if output_video is None:
        return "Tidak ditemukan file hasil YOLO."

    #Convert ke .mp4 jika perlu
    if output_video.endswith('.avi'):
        mp4_path = output_video.replace('.avi', '.mp4')
        ffmpeg.input(output_video).output(mp4_path, vcodec='libx264', acodec='aac').run(overwrite_output=True)
        output_video = mp4_path

    return output_video


#Custom CSS: navy background + white button
custom_css = """
body, .gradio-container {
    background-color: #001F3F !important;
}
h1, h2, h3, p, label {
    color: #ECF0F1 !important;
    font-family: 'Poppins', sans-serif;
}
button, .gr-button {
    background-color: white !important;
    color: #001F3F !important;
    border: 2px solid white !important;
    font-weight: bold !important;
    border-radius: 10px !important;
    font-size: 16px !important;
    transition: 0.3s;
}
button:hover, .gr-button:hover {
    background-color: #5DADE2 !important;
    color: white !important;
    border: 2px solid #5DADE2 !important;
}
"""

#Bangun UI Gradio (pakai CSS di atas)
demo = gr.Interface(
    fn=detect_video,
    inputs=gr.Video(label="Upload your satellite footage below"),
    outputs=gr.Video(label="🛰️ Detection Result"),
    title="Satellite Detection",
    description="Upload your satellite footage and let the AI model detect objects automatically. Supported formats: .mp4, .avi, etc.",
    css=custom_css
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59c3422db02a132cdb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Deploy Gradio

In [6]:
import os
import shutil
import ffmpeg
import gradio as gr
from ultralytics import YOLO

# Load model (pastikan best.pt ada di folder yang sama)
model = YOLO("best.pt")

def detect_video(video_path):
    # Buat nama file aman
    safe_name = os.path.basename(video_path).replace(" ", "_").replace("(", "").replace(")", "")
    safe_path = os.path.join(safe_name)
    shutil.copy(video_path, safe_path)

    # Jalankan YOLO
    model.predict(
        source=safe_path,
        save=True,
        project='runs/detect',
        name='gradio_output',
        exist_ok=True
    )

    # Cari hasil video
    folder = 'runs/detect/gradio_output'
    output_video = None
    for f in os.listdir(folder):
        if f.endswith('.avi') or f.endswith('.mp4'):
            output_video = os.path.join(folder, f)
            break

    if output_video is None:
        return "Tidak ditemukan file hasil YOLO."

    # Convert .avi → .mp4 biar bisa diplay
    if output_video.endswith('.avi'):
        mp4_path = output_video.replace('.avi', '.mp4')
        ffmpeg.input(output_video).output(mp4_path, vcodec='libx264', acodec='aac').run(overwrite_output=True)
        output_video = mp4_path

    return output_video


#Custom CSS (navy background + white button)
custom_css = """
body, .gradio-container {
    background-color: #001F3F !important;
}
h1, h2, h3, p, label {
    color: #ECF0F1 !important;
    font-family: 'Poppins', sans-serif;
}
button, .gr-button {
    background-color: white !important;
    color: #001F3F !important;
    border: 2px solid white !important;
    font-weight: bold !important;
    border-radius: 10px !important;
    font-size: 16px !important;
    transition: 0.3s;
}
button:hover, .gr-button:hover {
    background-color: #5DADE2 !important;
    color: white !important;
    border: 2px solid #5DADE2 !important;
}
"""

#Gradio app
demo = gr.Interface(
    fn=detect_video,
    inputs=gr.Video(label="Upload your satellite footage below"),
    outputs=gr.Video(label="Detection Result"),
    title="Satellite Detection",
    description="Upload your satellite footage and let the AI model detect objects automatically. Supported formats: .mp4, .avi, etc.",
    css=custom_css
)


FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'